# Lezione 43 — LoRA vs full fine-tuning: il confronto

> **Modulo:** LoRA e adattamento efficiente · **Tempo stimato:** 30 minuti
> **Prerequisiti:** Lezione 40 (LoRA da zero).
>
> Obiettivo pratico unico: confrontare **davvero** full fine-tuning e LoRA sullo
> stesso compito — parametri addestrati e perdita finale — per capire il
> compromesso.

## Teoria essenziale

LoRA promette *quasi* la qualita' del full fine-tuning con una frazione minima
dei parametri addestrati. "Quasi" quanto? Dipende dal rango $r$: piu' alto,
piu' l'adapter si avvicina al full fine-tuning (Lezione 39), ma piu' parametri.
Misuriamolo su un compito controllato dove l'aggiornamento vero e' di **rango
basso** (rango 3) — la premessa di LoRA (Lezione 39): quando $\Delta W$ e'
davvero a rango basso, un adapter con $r$ sufficiente puo' raggiungere il full
fine-tuning.

In [1]:
import numpy as np

rng = np.random.default_rng(43)

d, k, rango_vero = 24, 16, 3
W0 = rng.normal(size=(d, k)) * 0.3
# aggiornamento vero di RANGO BASSO (3) = prodotto di due fattori piccoli
Delta = rng.normal(size=(d, rango_vero)) @ rng.normal(size=(rango_vero, k)) * 0.5
W_target = W0 + Delta
X = rng.normal(size=(256, d))
Y = X @ W_target

def perdita(W):
    return np.mean((X @ W - Y) ** 2)

# 1) FULL fine-tuning: addestra tutta la matrice d x k
W = W0.copy()
for _ in range(400):
    W -= 0.05 * (2.0 / len(X)) * X.T @ (X @ W - Y)
full = {"param": d * k, "perdita": round(perdita(W), 5)}
print("full fine-tuning:", full)

full fine-tuning: {'param': 384, 'perdita': np.float64(0.0)}


In [2]:
def allena_lora(r, passi=1500, lr=0.01):
    A = rng.normal(size=(r, k)) * 0.01
    B = np.zeros((d, r))
    s = 1.0 / r
    for _ in range(passi):
        Weff = W0 + s * (B @ A)
        dW = (2.0 / len(X)) * X.T @ (X @ Weff - Y)
        B -= lr * s * (dW @ A.T)
        A -= lr * s * (B.T @ dW)
    return {"param": r * (d + k), "perdita": round(perdita(W0 + s * (B @ A)), 5)}

print(f"{'strategia':<16}{'param':>8}{'perdita':>12}")
print(f"{'full':<16}{full['param']:>8}{full['perdita']:>12}")
risultati = {"full": full}
for r in [1, 2, 3, 4, 8]:
    res = allena_lora(r)
    risultati[f"lora_r{r}"] = res
    print(f"{'lora r=' + str(r):<16}{res['param']:>8}{res['perdita']:>12}")

strategia          param     perdita
full                 384         0.0
lora r=1              40     9.22074
lora r=2              80      3.1284
lora r=3             120         0.0
lora r=4             160         0.0
lora r=8             320         0.0


Leggi la tabella: l'aggiornamento vero e' di rango 3, quindi salendo fino a
$r=3$ la perdita di LoRA crolla vicino a quella del full fine-tuning, ma con
pochissimi parametri. Oltre $r=3$ il guadagno e' marginale: aggiungere capacita'
non serve piu' — la scelta pratica e' il rango **piu' piccolo** che basta.

## Il progetto: Memory AI Lab, passo 43

Impacchetto il confronto come funzione che, dato un budget di qualita' (perdita
massima accettabile), sceglie il rango **piu' piccolo** che lo rispetta — la
decisione tipica quando si adatta il modello del progetto.

In [3]:
def scegli_rango(risultati, perdita_max):
    candidati = sorted((int(k.rsplit('_r', 1)[1]), v['perdita'])
                       for k, v in risultati.items() if k.startswith('lora_r'))
    for r, L in candidati:
        if L <= perdita_max:
            return r
    return None

soglia = 0.05     # perdita massima accettabile (assoluta) per questo compito
r_scelto = scegli_rango(risultati, soglia)
assert r_scelto is not None and r_scelto in (1, 2, 3, 4, 8)
# poiche' l'aggiornamento vero e' di rango 3, ci aspettiamo che basti r=3
print(f"soglia perdita <= {soglia} -> rango piu' piccolo adeguato: r={r_scelto}")

soglia perdita <= 0.05 -> rango piu' piccolo adeguato: r=3


## Riepilogo (max 8 punti)

1. LoRA punta a *quasi* la qualita' del full fine-tuning con pochi parametri.
2. Il rango $r$ regola il compromesso qualita'/costo.
3. Piu' $r$ → perdita piu' vicina al full, ma piu' parametri.
4. Su un compito controllato lo si misura direttamente.
5. Esiste un $r$ oltre cui il guadagno non giustifica il costo.
6. La scelta pratica: il rango **piu' piccolo** sotto una soglia di qualita'.
7. Il confronto e' onesto solo a parita' di compito e di passi.
8. Full fine-tuning resta il tetto di riferimento.

## Quiz

1. Perche' aumentare $r$ avvicina LoRA al full fine-tuning?
2. Qual e' il rischio di scegliere $r$ troppo alto?
3. Cosa rappresenta il full fine-tuning nel confronto?

*(Risposte: 1. piu' capacita' per approssimare $\Delta W$; 2. si perde il
vantaggio di efficienza avvicinandosi al costo del full; 3. il tetto di qualita'
di riferimento a cui LoRA si confronta.)*